# 04 - Datos geoespaciales de las calles de Albacete
En este Notebook se añade la información geoespacial (geometría de cada calle) a los datos limpios de Albacete obtenidos en el Notebook 02.

Los objetivos principales son:
* Cargar el archivo `albacete_viario_accesibilidad.json` (GeoJSON, [link](https://ide.ayto-albacete.es/geoserver/ws_albacete/wfs?service=WFS&version=1.1.0&request=GetFeature&outputFormat=json&typeName=albacete_viario_accesibilidad)) y comprobar su estructura.
* Comprobar que los identificadores coinciden con los del conjunto de datos limpio del 0.02.
* Unir la geometría a los datos limpios, tratando las calles con varios tramos o sin geometría.
* Visualizar el mapa de las calles y de su puntuación de accesibilidad.
* Visualizar los barrios, cargarlos en el conjunto de datos limpio.
* Guardar el conjunto de datos con geometría.

## Carga de datos

In [ ]:
import contextily as ctx
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd

from urban_accessibility_ml.config import (
    RAW_DATA_DIR,
    INTERIM_DATA_DIR,
)

ALBACETE_DIR = RAW_DATA_DIR / "albacete"
INTERIM_ALBACETE_DIR = INTERIM_DATA_DIR / "albacete"

from functools import partial
from urban_accessibility_ml.plots import save_figure, use_style, CMAP_ACC

use_style()
save_figure = partial(save_figure, folder="0.04-albacete-geoespacial", formats=('png',))

Cargamos los datos limpios de Albacete (Notebook 02) y el archivo con la geometría.

In [ ]:
df_alb_streets = pd.read_csv(INTERIM_ALBACETE_DIR / "albacete_streets_cleaned.csv")
gdf_alb_raw = gpd.read_file(ALBACETE_DIR / "albacete_viario_accesibilidad.json")

print(df_alb_streets.shape, gdf_alb_raw.shape)
print("CRS:", gdf_alb_raw.crs)
print(gdf_alb_raw.geom_type.value_counts())
display(gdf_alb_raw[["referencia", "num_calle", "nombre", "accglobal", "geometry"]].head())

## Comprobación de los identificadores
El archivo geoespacial contiene las mismas columnas del dataset original (en minúsculas y sin `ñ`), más la columna `geometry`. Comprobamos si `referencia` y `num_calle` se corresponden con `REFER` y `NUM_CALLE`.

In [ ]:
ids_geo = set(gdf_alb_raw["referencia"])
ids_clean = set(df_alb_streets["REFER"])

print("Calles en el CSV limpio:", len(ids_clean))
print("Calles distintas en el GeoJSON:", len(ids_geo))
print("En común:", len(ids_geo & ids_clean))
print("Solo en GeoJSON:", len(ids_geo - ids_clean))
print("Solo en CSV (sin geometría):", len(ids_clean - ids_geo))

# Check that REFER and NUM_CALLE identify the same street in both sources
check = gdf_alb_raw.merge(df_alb_streets, left_on="referencia", right_on="REFER")
print("NUM_CALLE coincide:", (check["num_calle"] == check["NUM_CALLE"]).all())
print("NOMBRE coincide:", (check["nombre"] == check["NOMBRE"]).all())
print("ACCGLOBAL coincide:", (check["accglobal"] == check["ACCGLOBAL"]).all())

Los identificadores de calles y referencias coinciden, aunque con un
Los identificadores son los mismos: todas las calles del GeoJSON existen en el dataset limpio, y nombre, número de calle y puntuación global coinciden. Pero hay dos diferencias:
* El GeoJSON tiene más filas (918) que calles distintas (904).
* Hay calles del dataset limpio que no tienen geometría.

Analizamos si hay filas duplicadas.

In [ ]:
# Streets that appear more than once in the GeoJSON
dup = gdf_alb_raw[gdf_alb_raw.duplicated("referencia", keep=False)].sort_values("referencia")
print("Filas duplicadas:", len(dup), "| calles afectadas:", dup["referencia"].nunique())
display(dup[["referencia", "nombre"]].assign(longitud_m=dup.length.round(1)).head(10))

# Streets without geometry
display(df_alb_streets[~df_alb_streets["REFER"].isin(ids_geo)][["REFER", "NOMBRE", "TIPOLACT"]])

Se unen las dos líneas de cada referencia de las calles repetidas en una única geometría (`MultiLineString`).

Hay 12 calles del conjunto de datos limpio sin geometría en el GeoJSON.

## Unión de los datos

In [ ]:
# One geometry per street: merge the lines that share the same REFER, and join them to the clean data (left join keeps streets without geometry)
gdf_geom = gdf_alb_raw.dissolve(by="referencia")[["geometry"]].reset_index()
gdf_alb_streets = gpd.GeoDataFrame(
    df_alb_streets.merge(gdf_geom, left_on="REFER", right_on="referencia", how="left").drop(columns="referencia"),
    geometry="geometry",
    crs=gdf_alb_raw.crs,
)

print(gdf_alb_streets.shape, "| Calles sin geometría:", gdf_alb_streets.geometry.isna().sum())
assert len(gdf_alb_streets) == len(df_alb_streets)

## Clases de accesibilidad (A, B y C)
Se divide la escala de la puntuación global (`ACCGLOBAL`, de 0 a 100) en tres tercios iguales y se asigna una clase a cada calle: **A** (azul, tercio superior), **B** (ámbar, tercio central) y **C** (naranja rojizo, tercio inferior).

In [ ]:
# Traffic light classes: thirds of the score scale (0-100); C < 1/3 <= B < 2/3 <= A
import numpy as np
SCALE = 100
from urban_accessibility_ml.plots import CLASS_COLORS
gdf_alb_streets["CLASE"] = pd.Series(
    np.select([gdf_alb_streets["ACCGLOBAL"] >= SCALE * 2 / 3, gdf_alb_streets["ACCGLOBAL"] >= SCALE / 3], ["A", "B"], default="C"),
    index=gdf_alb_streets.index,
).astype(pd.CategoricalDtype(["A", "B", "C"], ordered=True))
gdf_alb_streets["CLASE"] = gdf_alb_streets["CLASE"].where(gdf_alb_streets["ACCGLOBAL"].notna())
display(gdf_alb_streets["CLASE"].value_counts().sort_index().to_frame("calles").assign(porcentaje=lambda d: (d["calles"] / d["calles"].sum() * 100).round(1)))

## Visualización
Vamos a utilizar la ortofoto del PNOA (servicio WMTS del IGN, `GoogleMapsCompatible`), para visualizar mejor el callejero.
Mostramos la puntuación global de accesibilidad (`ACCGLOBAL`).

In [ ]:
PNOA_URL = (
    "https://www.ign.es/wmts/pnoa-ma?request=GetTile&service=WMTS&VERSION=1.0.0"
    "&Layer=OI.OrthoimageCoverage&Style=default&Format=image/jpeg"
    "&TileMatrixSet=GoogleMapsCompatible&TileMatrix={z}&TileRow={y}&TileCol={x}"
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
gdf_alb_streets.plot(column="ACCGLOBAL", cmap=CMAP_ACC, linewidth=1.5, legend=True, ax=ax,
                     legend_kwds={"label": "ACCGLOBAL"})
ctx.add_basemap(ax, source=PNOA_URL, crs=gdf_alb_streets.crs, attribution="PNOA © IGN")
ax.set_title("Albacete: puntuación global de accesibilidad por calle")
ax.set_axis_off()
save_figure("mapa_puntuacion_global_por_calle")
plt.show()

Mapa con las clases A, B y C.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
gdf_alb_streets.plot(color=gdf_alb_streets["CLASE"].map(CLASS_COLORS), linewidth=1.5, ax=ax)
ax.legend(handles=[Line2D([0], [0], color=c, lw=3, label=f"Clase {k}") for k, c in CLASS_COLORS.items()], loc="lower right")
ctx.add_basemap(ax, source=PNOA_URL, crs=gdf_alb_streets.crs, attribution="PNOA © IGN")
ax.set_title("Albacete: clase de accesibilidad por calle")
ax.set_axis_off()
save_figure("mapa_clase_de_accesibilidad")
plt.show()

Mapa por tipología de la calle (`TIPOLACT`), también sobre el PNOA.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
gdf_alb_streets.assign(TIPOLACT=gdf_alb_streets["TIPOLACT"].astype(str)).plot(
    column="TIPOLACT", categorical=True, linewidth=1.5, legend=True, ax=ax,
    legend_kwds={"loc": "lower right", "fontsize": 8},
)
ctx.add_basemap(ax, source=PNOA_URL, crs=gdf_alb_streets.crs, attribution="PNOA © IGN")
ax.set_title("Albacete: tipología de calle")
ax.set_axis_off()
save_figure("mapa_tipologia_de_calle")
plt.show()

## Ejemplos de calles por clase
Para ver mejor la clasificación sobre la ortofoto, se elige una calle de cada clase (A, B y C).

La calle elegida se dibuja más gruesa, y alrededor se muestran las calles vecinas con su clase.

In [ ]:
# One example street per class: the one whose length is closest to the median length of its class - AI generated
streets_geo = gdf_alb_streets.dropna(subset=["geometry"])
streets_geo = streets_geo.assign(length_m=streets_geo.length)
examples = {}
for cls in ["A", "B", "C"]:
    group = streets_geo[streets_geo["CLASE"] == cls]
    examples[cls] = (group["length_m"] - group["length_m"].median()).abs().idxmin()

fig, axes = plt.subplots(1, 3, figsize=(18, 6.5))
for ax, (cls, idx) in zip(axes, examples.items()):
    street = streets_geo.loc[idx]
    minx, miny, maxx, maxy = street.geometry.bounds
    half = max(maxx - minx, maxy - miny) / 2 + 150  # square view around the street, with a margin of 150 m
    cx, cy = (minx + maxx) / 2, (miny + maxy) / 2
    ax.set_xlim(cx - half, cx + half)
    ax.set_ylim(cy - half, cy + half)
    streets_geo.plot(color=streets_geo["CLASE"].map(CLASS_COLORS), linewidth=2, alpha=0.8, ax=ax)
    streets_geo.loc[[idx]].plot(color=CLASS_COLORS[cls], linewidth=6, ax=ax)
    streets_geo.loc[[idx]].plot(color="black", linewidth=1, linestyle="--", ax=ax)
    ctx.add_basemap(ax, source=PNOA_URL, crs=gdf_alb_streets.crs, attribution="PNOA © IGN")
    ax.set_title(f"Clase {cls}: {street['NOMBRE']} ({street['REFER']})\nACCGLOBAL: {street['ACCGLOBAL']:.0f} | longitud: {street['length_m']:.0f} m")
    ax.set_axis_off()
plt.tight_layout()
save_figure("ejemplos_de_calle_por_clase")
plt.show()

Mismas calles de ejemplo, con más zoom, para ver cómo es el entorno de la calle sobre la ortofoto.

In [ ]:
# Close-up of a short section of each example street (80 m x 80 m around the midpoint of the street) - AI generated
HALF_VIEW = 40  # meters

fig, axes = plt.subplots(1, 3, figsize=(18, 6.5))
for ax, (cls, idx) in zip(axes, examples.items()):
    street = streets_geo.loc[idx]
    mid = street.geometry.interpolate(0.5, normalized=True)
    ax.set_xlim(mid.x - HALF_VIEW, mid.x + HALF_VIEW)
    ax.set_ylim(mid.y - HALF_VIEW, mid.y + HALF_VIEW)
    streets_geo.plot(color=streets_geo["CLASE"].map(CLASS_COLORS), linewidth=4, alpha=0.8, ax=ax)
    streets_geo.loc[[idx]].plot(color=CLASS_COLORS[cls], linewidth=8, ax=ax)
    streets_geo.loc[[idx]].plot(color="black", linewidth=1.5, linestyle="--", ax=ax)
    ctx.add_basemap(ax, source=PNOA_URL, crs=gdf_alb_streets.crs, attribution="PNOA © IGN")
    ax.set_title(f"Clase {cls}: {street['NOMBRE']} ({street['REFER']})\nACCGLOBAL: {street['ACCGLOBAL']:.0f} | zoom: tramo de {2 * HALF_VIEW} m")
    ax.set_axis_off()
plt.tight_layout()
save_figure("ejemplos_de_calle_por_clase_zoom")
plt.show()

## Zonas
Las letras de `REFER` identifican la zona de cada calle (`ZONA`, 27 zonas diferentes en total).

Vamos a dibujar las áreas de las zonas, para tratar de identificar si son barrios.

In [ ]:
# Approximate area of each zone: convex hull of its streets with a 100 m margin - AI generated
zones = gpd.GeoDataFrame(
    gdf_alb_streets.dropna(subset=["geometry"]).dissolve(by="ZONA").buffer(100).convex_hull.rename("geometry").reset_index(),
    geometry="geometry", crs=gdf_alb_streets.crs,
)

fig, ax = plt.subplots(figsize=(12, 12))
colors = plt.get_cmap("tab20")(np.arange(len(zones)) % 20)
zones.plot(ax=ax, color=colors, alpha=0.3, edgecolor="none")
zones.boundary.plot(ax=ax, color="black", linestyle=":", linewidth=1.5)
streets_geo = gdf_alb_streets.dropna(subset=["geometry"])
streets_geo.plot(color=streets_geo["CLASE"].map(CLASS_COLORS), linewidth=1.5, ax=ax)
for _, zone in zones.iterrows():
    point = zone.geometry.representative_point()
    ax.annotate(zone["ZONA"], (point.x, point.y), ha="center", va="center", fontsize=9, color="black",
                bbox=dict(boxstyle="round,pad=0.25", facecolor="white", edgecolor="none", alpha=0.75))
ctx.add_basemap(ax, source=PNOA_URL, crs=gdf_alb_streets.crs, attribution="PNOA © IGN")
ax.set_title("Albacete: zonas (según las letras de REFER)")
ax.set_axis_off()
save_figure("mapa_de_zonas")
plt.show()

## Barrios de Albacete
Como se puede observar, las zonas están próximas geográficamente, por lo que vamos a tratar de identificar los barrios.

En la web https://aytoab365-my.sharepoint.com/personal/ayuntamiento_albacete_es/_layouts/15/onedrive.aspx?id=%2Fpersonal%2Fayuntamiento%5Falbacete%5Fes%2FDocuments%2FCartografia%2FCARTOTECA%20PUBLICACI%C3%93N%20WEB%2FDIVISIONES%20ADMINISTRATIVAS&viewid=84151e61%2Da4e3%2D4d86%2D8126%2Da7ed3ca7ab11&ga=1 del ayuntamiento de Albacete se puede obtener la información de los barrios (`2016MapaGeneralBarrios.pdf`), aunque solo en PDF e imágenes.

Como cada `ZONA` parece contener las iniciales del barrio (`CÑ` con Cañicas, `PL` con El Pilar, `SE` con Sepulcro-Bolera...), comparando manualmente, se ha podido relacionar las zonas con los barrios.

Aunque no se han encontrado los valores para las zonas X e Y.

In [ ]:
ZONA_TO_BARRIO = {
    "CA": "Carretas-Huerta Marzo", "CÑ": "Cañicas-Canal de María Cristina", "ES": "La Estrella",
    "FE": "Feria", "FR": "Franciscanos", "FT": "Fátima", "HF": "Hermanos Falcó", "HS": "Hospital",
    "ID": "Industria", "LL": "Los Llanos", "MD": "Medicina", "MI": "La Milagrosa", "PA": "La Pajarita",
    "PL": "El Pilar", "PM": "Pedro Lamata", "PQ": "Parque Sur", "PS": "Polígono de San Antón",
    "SA": "San Antonio Abad", "SE": "Sepulcro-Bolera", "SM": "San Pedro-Mortero", "SP": "San Pablo",
    "ST": "Santa Teresa", "UV": "Universidad", "VC": "Villacerrada-Centro", "VE": "Vereda",
}

# Number of streets per barrio
gdf_alb_streets["BARRIO"] = gdf_alb_streets["ZONA"].map(ZONA_TO_BARRIO)
display(gdf_alb_streets["BARRIO"].value_counts().to_frame("calles"))

print("Calles sin barrio (zonas X e Y):", gdf_alb_streets["BARRIO"].isna().sum())
display(gdf_alb_streets.loc[gdf_alb_streets["BARRIO"].isna(), ["NOMBRE", "ZONA"]].sort_values(["ZONA", "NOMBRE"]))

In [ ]:
# Streets of the zones X and Y, one color per zone
ZONE_COLORS = {"X": "#e41a1c", "Y": "#1f78b4"}
streets_xy = gdf_alb_streets[gdf_alb_streets["ZONA"].isin(ZONE_COLORS)].dropna(subset=["geometry"])

fig, ax = plt.subplots(figsize=(12, 12))
streets_xy.plot(color=streets_xy["ZONA"].map(ZONE_COLORS), linewidth=2.5, ax=ax)
ax.legend(handles=[Line2D([0], [0], color=color, lw=3, label=f"Zona {zone}") for zone, color in ZONE_COLORS.items()], loc="lower right")
ctx.add_basemap(ax, source=PNOA_URL, crs=gdf_alb_streets.crs, attribution="PNOA © IGN")
ax.set_title("Albacete: calles de las zonas X e Y (sin barrio)")
ax.set_axis_off()
save_figure("mapa_de_calles_de_las_zonas_x_e_y_sin_barrio")
plt.show()

Según se puede comprobar, las calles de la zona X, parecen carreteras principales de acceso a la ciudad.
Mientras que la zona Y parecen las calles más largas de la ciudad.

## Mapa de calor por barrio
Para cada barrio se calcula la media, la mediana y la clase de la puntuación global (`ACCGLOBAL`) de sus calles. Se muestra un mapa mapa de calor por barrio, del rojo fuerte (valor más bajo) al verde fuerte (valor más alto), ajustada al rango de los barrios para destacar sus diferencias.

La mayoría de barrios tienen clase B o C, por lo que sería difícil distinguirlos solo por eso.

In [ ]:
# Approximate area of each neighbourhood: convex hull of its streets with a 100 m margin - AI generated
barrios = gpd.GeoDataFrame(
    gdf_alb_streets.dropna(subset=["geometry"]).dissolve(by="BARRIO").buffer(100).convex_hull.rename("geometry").reset_index(),
    geometry="geometry", crs=gdf_alb_streets.crs,
)

STAT = "median"  # "mean" or "median"
scores = gdf_alb_streets.groupby("BARRIO")["ACCGLOBAL"].agg(["mean", "median", "count"])
scores["CLASE"] = np.select([scores[STAT] >= SCALE * 2 / 3, scores[STAT] >= SCALE / 3], ["A", "B"], default="C")
display(scores.sort_values(STAT, ascending=False).round(2))

areas = barrios.merge(scores, left_on="BARRIO", right_index=True)
streets_geo = gdf_alb_streets.dropna(subset=["geometry"])

fig, ax = plt.subplots(figsize=(14, 14))
# Continuous scale from strong red (lowest value) to strong green (highest), adjusted to the range of the neighbourhoods
norm = Normalize(vmin=scores[STAT].min(), vmax=scores[STAT].max())
cmap = CMAP_ACC
areas.plot(ax=ax, color=[cmap(norm(value)) for value in areas[STAT]], alpha=0.7)
areas.boundary.plot(ax=ax, color="black", linestyle=":", linewidth=1.5)
streets_geo.plot(color="black", linewidth=0.5, alpha=0.5, ax=ax)
for _, area in areas.iterrows():
    point = area.geometry.representative_point()
    ax.annotate(f"{area['BARRIO']}\n{area[STAT]:.0f}", (point.x, point.y), ha="center", va="center", fontsize=9, color="black",
                bbox=dict(boxstyle="round,pad=0.25", facecolor="white", edgecolor="none", alpha=0.75))
fig.colorbar(ScalarMappable(norm=norm, cmap=cmap), ax=ax, shrink=0.5, label=f"{'Media' if STAT == 'mean' else 'Mediana'} de ACCGLOBAL")
ctx.add_basemap(ax, source=PNOA_URL, crs=gdf_alb_streets.crs, attribution="PNOA © IGN")
ax.set_title(f"Albacete: {'media' if STAT == 'mean' else 'mediana'} de ACCGLOBAL por barrio")
ax.set_axis_off()
save_figure("mapa_de_calor_por_barrio")
plt.show()

Se observa que el barrio con mayor accesibilidad es el del centro, probablemente por un número más elevado de calles peatonales.

## Guardado del conjunto de datos con geometría
Se guarda en `data/interim/albacete` en formato GeoJSON.

In [ ]:
gdf_alb_streets.to_file(INTERIM_ALBACETE_DIR / "albacete_streets_geo.geojson", driver="GeoJSON")

Se añade también el barrio (`BARRIO`) al conjunto de datos limpio (CSV) de Albacete, sin la geometría. Las zonas `X` e `Y`, que no tienen barrio, conservan su letra como valor de `BARRIO`.

In [ ]:
df_alb_streets["BARRIO"] = df_alb_streets["ZONA"].map(ZONA_TO_BARRIO).fillna(df_alb_streets["ZONA"])  # zones X and Y have no barrio, they keep their letter
df_alb_streets["TRAMO"] = df_alb_streets["TRAMO"].astype("Int64")  # reading the CSV turns it into float
df_alb_streets.to_csv(INTERIM_ALBACETE_DIR / "albacete_streets_cleaned.csv", index=False)